# Lab 2.4 – Reproducibility and Logging Basics

This notebook demonstrates a reproducible ETL workflow. It configures Python logging to capture console and file output, records environment details, and produces simple metrics from the Week 1 restaurant datasets. When pandas is unavailable (such as in a restricted execution environment), the ETL steps fall back to pure-Python data handling so the notebook remains runnable.

In [ ]:
import csv
import json
import logging
import os
import platform
import random
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

# Optional third-party imports
try:
    import numpy as np
except ModuleNotFoundError:  # fallback when numpy is not installed
    np = None
try:
    import pandas as pd
except ModuleNotFoundError:  # fallback when pandas is not installed
    pd = None

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

# Load configuration with fallbacks
BASE_DIR = Path.cwd()
CONFIG_PATH = BASE_DIR / "config.yaml"
default_paths = {"data_dir": "data", "logs_dir": "logs", "output_dir": "etl_pipeline"}
paths = {"paths": default_paths}
if CONFIG_PATH.exists() and yaml is not None:
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        loaded = yaml.safe_load(f) or {}
    paths.update(loaded)
merged_paths = {**default_paths, **paths.get("paths", {})}

# Ensure directories exist
logs_dir = BASE_DIR / merged_paths["logs_dir"]
logs_dir.mkdir(parents=True, exist_ok=True)
output_dir = BASE_DIR / merged_paths["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)

# Configure logging
run_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
log_file = logs_dir / f"run_{run_ts}.log"
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
root_logger.handlers.clear()

stream_handler = logging.StreamHandler(stream=sys.stdout)
stream_handler.setFormatter(formatter)
file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(formatter)

root_logger.addHandler(stream_handler)
root_logger.addHandler(file_handler)

logging.info("Starting reproducibility and logging ETL run")
logging.info("Python runtime: %s", platform.python_version())
logging.info("Platform: %s", platform.platform())
logging.info("Active paths: %s", merged_paths)
print(f"Log file created at: {log_file}")

In [ ]:
# Reproducibility seeds
os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
if np is not None:
    np.random.seed(0)
logging.info("Seeds initialized for reproducibility")

In [ ]:
# Capture current environment
requirements_path = BASE_DIR / "requirements.txt"
import subprocess
subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, stdout=requirements_path.open("w"))
logging.info("Wrote pip freeze to %s", requirements_path)

In [ ]:
# Hash input data files
import hashlib

def sha256sum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

data_dir = BASE_DIR / merged_paths["data_dir"]
hashes = {}
for csv_path in data_dir.glob("*.csv"):
    digest = sha256sum(csv_path)
    hashes[csv_path.name] = digest
    logging.info("SHA-256 for %s: %s", csv_path.name, digest)

hash_path = BASE_DIR / "data_hashes.json"
with open(hash_path, "w", encoding="utf-8") as f:
    json.dump(hashes, f, indent=2)
logging.info("Saved data hashes to %s", hash_path)

In [ ]:
# Load and clean data with pandas when available, otherwise use pure Python

data_dir = BASE_DIR / merged_paths["data_dir"]
orders_path = data_dir / "order_details.csv"
menu_path = data_dir / "menu_items.csv"

if pd is not None:
    orders = pd.read_csv(orders_path)
    menu = pd.read_csv(menu_path)

    orders["order_time"] = pd.to_datetime(orders["order_time"], errors="coerce")
    orders["customer_name"] = orders["customer_name"].str.strip()
    orders["quantity"] = orders["quantity"].astype(int)

    menu["item_name"] = menu["item_name"].str.strip()
    menu["category"] = menu["category"].str.title().str.strip()
    menu["price"] = menu["price"].astype(float)

    etl_df = orders.merge(menu, how="left", left_on="item_id", right_on="item_id")
    etl_df["line_total"] = etl_df["price"] * etl_df["quantity"]
    etl_df["order_hour"] = etl_df["order_time"].dt.hour
    missing = set(
        [
            "order_id",
            "order_time",
            "customer_name",
            "item_id",
            "item_name",
            "category",
            "price",
            "quantity",
            "line_total",
            "order_hour",
        ]
    ) - set(etl_df.columns)
    assert not missing, f"Missing columns after merge: {missing}"
    assert not etl_df.empty, "Merged dataset is empty"
    logging.info("Loaded and merged %d rows with pandas", len(etl_df))
else:
    # Pure Python fallback
    def load_csv(path):
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            return [row for row in reader]

    orders = load_csv(orders_path)
    menu = load_csv(menu_path)
    menu_lookup = {int(row["item_id"]): row for row in menu}

    etl_rows = []
    for row in orders:
        item_id = int(row["item_id"])
        merged = {
            "order_id": int(row["order_id"]),
            "order_time": row["order_time"],
            "customer_name": row["customer_name"].strip(),
            "item_id": item_id,
            "quantity": int(row["quantity"]),
        }
        menu_row = menu_lookup.get(item_id, {})
        merged["item_name"] = menu_row.get("item_name", "").strip()
        merged["category"] = menu_row.get("category", "").title().strip()
        merged["price"] = float(menu_row.get("price", 0.0))
        merged["line_total"] = merged["price"] * merged["quantity"]
        merged["order_hour"] = int(row["order_time"].split()[1].split(":")[0])
        etl_rows.append(merged)

    etl_df = etl_rows
    logging.info("Loaded and merged %d rows with pure Python fallback", len(etl_df))

if pd is not None:
    display(etl_df.head())
else:
    etl_df[:5]

In [ ]:
# Metrics
metric_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

if pd is not None:
    TopItems = (
        etl_df.groupby(["item_id", "item_name"], dropna=False)
        .agg(total_quantity=("quantity", "sum"))
        .reset_index()
        .sort_values("total_quantity", ascending=False)
        .head(5)
    )

    RevenueByCategory = (
        etl_df.groupby("category", dropna=False)
        .agg(total_revenue=("line_total", "sum"))
        .reset_index()
        .sort_values("total_revenue", ascending=False)
    )

    BusiestHour = (
        etl_df.groupby("order_hour", dropna=False)
        .agg(order_count=("order_id", "nunique"))
        .reset_index()
        .sort_values("order_count", ascending=False)
    )

    stacked_metrics = pd.concat(
        [
            TopItems.assign(metric="top_items"),
            RevenueByCategory.assign(metric="revenue_by_category"),
            BusiestHour.assign(metric="busiest_hour"),
        ],
        ignore_index=True,
    )
else:
    # Pure Python metrics
    top_counter = Counter()
    revenue_by_cat = defaultdict(float)
    order_counter = Counter()

    for row in etl_df:
        top_counter[(row["item_id"], row["item_name"])] += row["quantity"]
        revenue_by_cat[row["category"]] += row["line_total"]
        order_counter[row["order_hour"]] += 1

    top_items = [
        {"item_id": item_id, "item_name": name, "total_quantity": qty}
        for (item_id, name), qty in top_counter.most_common(5)
    ]
    revenue_rows = [
        {"category": cat, "total_revenue": rev}
        for cat, rev in sorted(revenue_by_cat.items(), key=lambda x: x[1], reverse=True)
    ]
    busiest_rows = [
        {"order_hour": hour, "order_count": count}
        for hour, count in sorted(order_counter.items(), key=lambda x: x[1], reverse=True)
    ]

    # Stack results
    stacked_metrics = []
    for row in top_items:
        stacked_metrics.append({**row, "metric": "top_items"})
    for row in revenue_rows:
        stacked_metrics.append({**row, "metric": "revenue_by_category"})
    for row in busiest_rows:
        stacked_metrics.append({**row, "metric": "busiest_hour"})

# Save metrics
metrics_path = output_dir / f"metrics_{metric_ts}.csv"
if pd is not None:
    stacked_metrics.to_csv(metrics_path, index=False)
    preview = stacked_metrics.head()
else:
    fieldnames = sorted({key for row in stacked_metrics for key in row.keys()})
    with open(metrics_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(stacked_metrics)
    preview = stacked_metrics[:5]

logging.info("Saved metrics to %s", metrics_path)
preview

The `metrics_*` CSV contains stacked results for all three metrics with a `metric` label column so you can filter per metric after loading.